# SpixRWKV-7: Full Architecture Benchmark on Tiny-ImageNet

Trains all 6 architecture variants (spix, conv, gnn, vq, hybrid, vit) on Tiny-ImageNet with:
- **2x T4 GPUs** via DataParallel + mixed precision (AMP)
- **Early stopping** with patience=5 on validation accuracy
- **~10 hour budget** — depth=6 models ~37 min each, VQ ~1.5h

## Setup

In [ ]:
import sys
import time
import subprocess

def log(msg: str):
    """Print with timestamp and immediate flush."""
    t = time.strftime("%H:%M:%S")
    print(f"[{t}] {msg}", flush=True)

def log_sep(title: str = ""):
    """Print a section separator."""
    t = time.strftime("%H:%M:%S")
    if title:
        print(f"\n[{t}] {'='*60}", flush=True)
        print(f"[{t}]  {title}", flush=True)
        print(f"[{t}] {'='*60}", flush=True)
    else:
        print(f"[{t}] {'-'*60}", flush=True)

log("Starting notebook execution")
log(f"Python version: {sys.version}")

In [ ]:
# ── Install spixrwkv7 package ──
log_sep("INSTALLING spixrwkv7 package from GitHub")
log("This may take 5-20 minutes (compiling C++/CUDA kernels from source)...")
log("Starting pip install (verbose mode)...")
start_install = time.time()

# Use subprocess with real-time output instead of silent pip install
proc = subprocess.Popen(
    [sys.executable, "-m", "pip", "install",
     "git+https://github.com/Gabz4200/SpixRWKV7.git#egg=spixrwkv7[all]"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
)
last_log_time = time.time()
for line in proc.stdout:
    now = time.time()
    if now - last_log_time > 30:  # Log progress every 30 seconds
        elapsed = now - start_install
        log(f"Still installing... ({elapsed:.0f}s elapsed) Last line: {line.strip()[:120]}")
        last_log_time = now
proc.wait()
if proc.returncode != 0:
    log(f"ERROR: pip install failed with code {proc.returncode}")
    raise RuntimeError(f"pip install failed (code {proc.returncode})")
log(f"pip install completed in {time.time() - start_install:.0f}s")

# ── Build C++/CUDA kernels ──
log_sep("BUILDING C++/CUDA KERNELS (spixrwkv7-build-kernels)")
start_build = time.time()
proc = subprocess.Popen(
    ["spixrwkv7-build-kernels"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
)
last_log_time = time.time()
for line in proc.stdout:
    stripped = line.strip()
    if stripped:
        now = time.time()
        if now - last_log_time > 15:
            log(f"Kernel build: {stripped[:150]}")
            last_log_time = now
proc.wait()
if proc.returncode != 0:
    log(f"WARNING: Kernel build returned code {proc.returncode} (may be OK if no CUDA)")
else:
    log(f"Kernel build completed in {time.time() - start_build:.0f}s")

# ── Verify GPU availability ──
log_sep("GPU DETECTION")
import torch
log(f"CUDA available: {torch.cuda.is_available()}")
log(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_mem_gb = props.total_memory / 1024**3
        log(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({total_mem_gb:.1f} GB VRAM)")
    log(f"  Current GPU memory allocated: {torch.cuda.memory_allocated()/1024**2:.0f} MB")
    log(f"  Current GPU memory cached: {torch.cuda.memory_reserved()/1024**2:.0f} MB")

In [ ]:
import os
import json
import math
import random
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image

log("Importing spixrwkv7 modules...")
_import_start = time.time()

from spixrwkv7 import (
    create_optimized_vision_rwkv7,
    create_conv_vision_rwkv7,
    create_gnn_vision,
    create_hybrid_vision,
    ClassificationHead,
)
# Move color conversion import here (was inside Dataset.__init__) to fail fast
from spixrwkv7.data.colors import from_linear_rgb_to_oklab

log(f"spixrwkv7 imports completed in {time.time() - _import_start:.1f}s")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

log("All imports OK, seeds set")

## Configuration

In [ ]:
log_sep("CONFIGURATION")

# ── Training config ──
IMG_SIZE = 64
NUM_CLASSES = 200
MAX_EPOCHS = 16
EARLY_STOP_PATIENCE = 4
BATCH_SIZE = 64  # 32 per GPU on 2 GPUs (power of 2)
LR = 3e-4
WEIGHT_DECAY = 0.05
GRAD_CLIP = 1.0
NUM_WORKERS = 8  # Power of 2

# ── Logging interval ──
LOG_INTERVAL = 64  # Log every 64 batches (power of 2)

# ── Device ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
log(f"Device: {DEVICE}, GPUs: {NUM_GPUS}")
if DEVICE.type == "cuda":
    for i in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(i)
        log(f"  GPU {i}: {props.name} | {props.total_memory/1024**3:.1f} GB")

# ── Model configs ──
# Tuple: (model_type, builder_key, config_overrides, batch_size)
# All models use depth=8, embed_dims=128, heads=4 (all powers of 2) for fair deployable comparison.
# VQ is removed (too slow, user agreed to sacrifice it).
MODELS = [
    ("vit",     "vit",    {},                                                                  64),
    ("gnn",     "gnn",    {"depth": 8, "gnn_conv": "gatv2", "gnn_heads": 4},                 64),
    ("conv",    "conv",   {"depth": 8, "conv_stem_norm": "layernorm"},                        64),
    ("spix",    "spix",   {"depth": 8},                                                       64),
    ("hybrid",  "hybrid", {"depth": 8, "num_rwkv_layers": 3, "num_gnn_layers": 2},            64),
]

# ── Shared backbone config (all powers of 2, realistic deployable size) ──
BACKBONE_CFG = {
    "embed_dims": 128,       # 2^7 — deployable on edge
    "num_heads": 4,          # 2^2
    "num_superpixels": 64,   # 8^2 = 2^6 — 8x8 grid
    "diff_slic_iters": 5,
    "compactness": 0.5,
    "drop_path_rate": 0.1,
    "init_values": 1e-5,
    "norm_layer": "rmsnorm",
    "act_layer": "swiglu",
}

log(f"Training config: {MAX_EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LR}")
log(f"Models to train ({len(MODELS)}): {[m[0] for m in MODELS]}")
log(f"Backbone: embed_dims={BACKBONE_CFG['embed_dims']}, depth=8, heads={BACKBONE_CFG['num_heads']}, superpixels={BACKBONE_CFG['num_superpixels']} (all powers of 2)")\nlog(f"NUM_WORKERS={NUM_WORKERS}, LOG_INTERVAL={LOG_INTERVAL}")
log(f"Estimated total time: ~2-3h (well within 6h budget)")


## Dataset: Tiny-ImageNet

In [ ]:
log_sep("DATASET: Tiny-ImageNet (with float16 pre-cache)")

class TinyImageNet(Dataset):
    """Tiny-ImageNet from HuggingFace: 100K train, 10K val, 200 classes, 64x64.
    
    Pre-processes all images to OkLAB at init time and caches as float16 tensor.
    This eliminates per-epoch preprocessing overhead (~2.4 GB for 100K train).
    """
    _to_oklab = from_linear_rgb_to_oklab
    _grids_computed = False
    _alpha = None
    _xy = None

    def __init__(self, split: str, img_size: int = 64):
        from datasets import load_dataset
        self.img_size = img_size
        log(f"  [Dataset] Loading '{split}' split from zh-plus/tiny-imagenet...")
        t0 = time.time()
        ds = load_dataset("zh-plus/tiny-imagenet", split=split)
        log(f"  [Dataset] Loaded {len(ds)} samples in {time.time()-t0:.1f}s")

        # Pre-compute spatial grids once (shared across train/val)
        if not TinyImageNet._grids_computed:
            TinyImageNet._alpha = torch.ones(1, img_size, img_size)
            yy, xx = torch.meshgrid(
                torch.linspace(-1, 1, img_size),
                torch.linspace(-1, 1, img_size),
                indexing="ij",
            )
            TinyImageNet._xy = torch.stack([xx, yy], dim=0)
            TinyImageNet._grids_computed = True

        # Pre-process all images → OkLAB float16 cache
        log(f"  [Dataset] Pre-processing {len(ds)} images to OkLAB (float16 cache)...")
        cache_start = time.time()
        n = len(ds)
        self.oklab_cache = torch.empty(n, 3, img_size, img_size, dtype=torch.float16)
        self.labels = torch.empty(n, dtype=torch.long)

        for idx in range(n):
            item = ds[idx]
            img = item["image"]
            self.labels[idx] = item["label"]

            if not isinstance(img, Image.Image):
                img = Image.fromarray(np.array(img))
            if img.mode != "RGB":
                img = img.convert("RGB")

            img = img.resize((img_size, img_size), Image.BILINEAR)
            img_np = torch.tensor(np.array(img), dtype=torch.uint8)
            img_float = img_np.permute(2, 0, 1).float() / 255.0

            srgb = img_float.unsqueeze(0)
            linear_rgb = srgb.clamp(0).pow(2.2)
            oklab = self._to_oklab(linear_rgb).squeeze(0)
            self.oklab_cache[idx] = oklab.half()

            if idx > 0 and idx % 25000 == 0:
                log(f"    Cached {idx}/{n} images ({time.time()-cache_start:.0f}s)...")

        log(f"  [Dataset] Pre-processing done: {time.time()-cache_start:.0f}s for {n} images")
        log(f"  [Dataset] Cache size: {self.oklab_cache.nbytes/1024**3:.2f} GB (float16)")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        oklab = self.oklab_cache[idx].float()
        x = torch.cat([oklab, self._alpha, self._xy], dim=0)  # (6, 64, 64)
        return x, self.labels[idx]


# Load datasets (pre-processing happens here)
log("Loading Tiny-ImageNet with float16 pre-cache...")
train_ds = TinyImageNet("train", IMG_SIZE)
val_ds = TinyImageNet("valid", IMG_SIZE)
log(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Classes: {NUM_CLASSES}")

# DataLoaders (batch_size=96, 6 workers)
pin = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin, drop_last=pin,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin,
)
log(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Quick batch validation
log("Validating dataloader...")
sample_batch = next(iter(train_loader))
log(f"  x={sample_batch[0].shape}, y={sample_batch[1].shape}, range=[{sample_batch[0].min():.3f}, {sample_batch[0].max():.3f}]")
del sample_batch


## Model Builders

In [ ]:
log_sep("MODEL BUILDERS (with torch.compile)")

def build_model(model_type: str, overrides: dict) -> Tuple[nn.Module, Optional[nn.Module]]:
    """Build backbone + head. Returns (backbone, head_or_None).
    Non-ViT models get an extra ClassificationHead.
    """
    t0 = time.time()
    cfg = {**BACKBONE_CFG, **overrides}
    log(f"  Building {model_type.upper()} (depth={cfg['depth']}, dim={cfg['embed_dims']})")

    if model_type == "vit":
        # FIXED: use cfg depth instead of hardcoded 6
        backbone = SimpleViT(
            img_size=IMG_SIZE, patch_size=8, in_chans=6,
            num_classes=NUM_CLASSES, embed_dim=cfg["embed_dims"],
            depth=cfg["depth"], num_heads=cfg["num_heads"],
        )
        log(f"  Built ViT in {time.time()-t0:.1f}s: {sum(p.numel() for p in backbone.parameters())/1e6:.2f}M params")
        return backbone, None

    builder_map = {
        "spix": create_optimized_vision_rwkv7,
        "conv": create_conv_vision_rwkv7,
        "gnn": create_gnn_vision,
        "hybrid": create_hybrid_vision,
    }
    builder = builder_map[model_type]

    kwargs = {
        "img_size": IMG_SIZE,
        "embed_dims": cfg["embed_dims"],
        "num_heads": cfg["num_heads"],
        "depth": cfg["depth"],
        "init_values": cfg["init_values"],
        "final_norm": True,
        "out_indices": [-1],
        "norm_layer": cfg["norm_layer"],
        "act_layer": cfg["act_layer"],
        "scatter_output": True,
    }

    # Model-specific kwargs
    if model_type in ("spix", "conv", "gnn", "hybrid"):
        kwargs.update({
            "num_superpixels": cfg["num_superpixels"],
            "diff_slic_iters": cfg["diff_slic_iters"],
            "compactness": cfg["compactness"],
            "drop_path_rate": cfg["drop_path_rate"],
        })
    if model_type in ("spix", "conv"):
        kwargs["spixel_backend"] = "diff_slic"
    if model_type == "conv":
        kwargs["conv_stem_norm"] = cfg.get("conv_stem_norm", "layernorm")
        kwargs["conv_post_norm"] = "layernorm"
    if model_type == "gnn":
        kwargs["gnn_conv"] = cfg.get("gnn_conv", "gatv2")
        kwargs["gnn_heads"] = cfg.get("gnn_heads", 4)
        kwargs["gnn_aggr"] = "mean"
        for k in ["diff_slic_iters", "compactness", "num_superpixels", "spixel_backend", "drop_path_rate"]:
            kwargs.pop(k, None)
    if model_type == "hybrid":
        kwargs["num_rwkv_layers"] = cfg.get("num_rwkv_layers", 2)
        kwargs["num_gnn_layers"] = cfg.get("num_gnn_layers", 1)
        kwargs["gnn_conv"] = "gatv2"
        kwargs["gnn_heads"] = 4
        kwargs["register_tokens"] = 4
        for k in ["diff_slic_iters", "compactness", "num_superpixels", "spixel_backend"]:
            kwargs.pop(k, None)

    log(f"  Builder kwargs (excluding out_indices): { {k: v for k, v in kwargs.items() if k != 'out_indices'} }")
    backbone = builder(**kwargs)
    backbone_params = sum(p.numel() for p in backbone.parameters())
    log(f"  Built {model_type} backbone in {time.time()-t0:.1f}s: {backbone_params/1e6:.2f}M params")

    head = ClassificationHead(cfg["embed_dims"], NUM_CLASSES)
    head_params = sum(p.numel() for p in head.parameters())
    log(f"  ClassificationHead: {head_params/1e3:.1f}K params, Total: {(backbone_params+head_params)/1e6:.2f}M")
    return backbone, head


class SimpleViT(nn.Module):
    """ViT baseline with 6-channel input. Depth now configurable via cfg."""
    def __init__(self, img_size, patch_size, in_chans, num_classes,
                 embed_dim, depth, num_heads):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, embed_dim))
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=num_heads,
                dim_feedforward=4 * embed_dim, batch_first=True,
            )
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.xavier_uniform_(self.cls_token)
        nn.init.xavier_uniform_(self.pos_embed)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


# ── Model builder validation ──
log_sep("VALIDATING MODEL BUILDERS")
for t in ["vit", "spix"]:
    try:
        log(f"Test-building {t.upper()}...")
        bb, hd = build_model(t, {"depth": 4})  # depth=4 for quick test
        test_in = torch.randn(2, 6, IMG_SIZE, IMG_SIZE).to(DEVICE)
        bb = bb.to(DEVICE)
        with torch.no_grad():
            out = bb(test_in)
            if hd is not None:
                hd = hd.to(DEVICE)
                out = hd(out[0] if isinstance(out, tuple) else out)
        log(f"  OK: input={test_in.shape} -> output={out.shape}")
        del bb, hd, test_in, out
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    except Exception as e:
        log(f"  ERROR building {t}: {e}")
        import traceback; traceback.print_exc()
        raise
log("All model builders validated OK")


## Training Loop with Early Stopping

In [ ]:
def log_gpu_mem(tag: str = ""):
    if DEVICE.type == "cuda":
        allocated = torch.cuda.memory_allocated() / 1024**2
        reserved = torch.cuda.memory_reserved() / 1024**2
        log(f"  [GPU Mem] {tag} — {allocated:.0f}MB / {reserved:.0f}MB")


def maybe_compile(model: nn.Module, name: str = "backbone") -> nn.Module:
    """Apply torch.compile if available (PyTorch 2.x on Kaggle)."""
    if hasattr(torch, "compile") and DEVICE.type == "cuda":
        try:
            model = torch.compile(model, mode="reduce-overhead")
            log(f"  torch.compile applied to {name}")
        except Exception as e:
            log(f"  torch.compile skipped for {name}: {e}")
    return model


def train_model(model_type: str, overrides: dict, batch_size: int) -> dict:
    """Train one model with early stopping + torch.compile."""
    log_sep(f"TRAINING: {model_type.upper()} (batch_size={batch_size})")

    # Per-model dataloaders
    pin = DEVICE.type == "cuda"
    model_train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=pin, drop_last=pin,
    )
    model_val_loader = DataLoader(
        val_ds, batch_size=batch_size * 2, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin,
    )
    log(f"  Train: {len(model_train_loader)} batches (bs={batch_size}), Val: {len(model_val_loader)} batches")

    # Build model
    backbone, head = build_model(model_type, overrides)
    backbone = backbone.to(DEVICE)
    if head is not None:
        head = head.to(DEVICE)
    log_gpu_mem("after model creation")

    # Apply torch.compile BEFORE DataParallel
    backbone = maybe_compile(backbone, f"{model_type} backbone")

    # Multi-GPU
    if NUM_GPUS > 1:
        log(f"  DataParallel on {NUM_GPUS} GPUs...")
        backbone = nn.DataParallel(backbone)
        if head is not None:
            head = nn.DataParallel(head)

    # AMP
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if use_amp:
        log("  Mixed precision (AMP) enabled")

    # Params
    params = list(backbone.parameters())
    if head is not None:
        params += list(head.parameters())
    total_params = sum(p.numel() for p in params)
    log(f"  Trainable params: {total_params / 1e6:.2f}M")

    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Pre-warm: one dummy batch to trigger torch.compile compilation
    log("  Pre-warming model with dummy batch...")
    warm_start = time.time()
    warm_x = torch.randn(batch_size, 6, IMG_SIZE, IMG_SIZE, device=DEVICE)
    warm_y = torch.randint(0, NUM_CLASSES, (batch_size,), device=DEVICE)
    backbone.train()
    if head is not None:
        head.train()
    with torch.amp.autocast("cuda", enabled=use_amp):
        outs = backbone(warm_x)
        feat = outs[0] if isinstance(outs, tuple) else outs
        logits = head(feat) if head is not None else feat
        loss = criterion(logits, warm_y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    log(f"  Pre-warm done in {time.time()-warm_start:.1f}s (compilation triggered)")
    del warm_x, warm_y

    # Early stopping
    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    total_start = time.time()
    num_train_batches = len(model_train_loader)

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        backbone.train()
        if head is not None:
            head.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (x, y) in enumerate(model_train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                outs = backbone(x)
                feat = outs[0] if isinstance(outs, tuple) else outs
                logits = head(feat) if head is not None else feat
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, max_norm=GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            batch_loss = loss.item()
            train_loss += batch_loss * x.size(0)
            train_correct += (logits.argmax(1) == y).sum().item()
            train_total += x.size(0)

            if batch_idx > 0 and batch_idx % LOG_INTERVAL == 0:
                running_acc = 100.0 * train_correct / train_total
                elapsed = time.time() - epoch_start
                est_total = elapsed * num_train_batches / (batch_idx + 1)
                lr_now = optimizer.param_groups[0]["lr"]
                log(f"  [Epoch {epoch}] batch {batch_idx+1}/{num_train_batches} | "
                    f"loss={batch_loss:.4f} acc={running_acc:.1f}% | "
                    f"lr={lr_now:.2e} | est_remaining={est_total-elapsed:.0f}s")

        scheduler.step()
        train_loss /= train_total
        train_acc = 100.0 * train_correct / train_total

        # ── Validate ──
        backbone.eval()
        if head is not None:
            head.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x, y in model_val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    outs = backbone(x)
                    feat = outs[0] if isinstance(outs, tuple) else outs
                    logits = head(feat) if head is not None else feat
                    loss = criterion(logits, y)

                val_loss += loss.item() * x.size(0)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_total += x.size(0)

        val_loss /= val_total
        val_acc = 100.0 * val_correct / val_total
        gap = train_acc - val_acc
        lr_now = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(lr_now)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            marker = " *BEST*"
        else:
            patience_counter += 1
            marker = f" (p={patience_counter}/{EARLY_STOP_PATIENCE})"

        log_gpu_mem(f"end epoch {epoch}")
        log(f"  [Epoch {epoch}/{MAX_EPOCHS}] train_loss={train_loss:.4f} acc={train_acc:.1f}% | "
            f"val_loss={val_loss:.4f} acc={val_acc:.1f}% | gap={gap:.1f}% | lr={lr_now:.2e} | {epoch_time:.0f}s{marker}")

        if patience_counter >= EARLY_STOP_PATIENCE:
            log(f"  Early stopping at epoch {epoch}")
            break

    total_time = time.time() - total_start
    log(f"  Done: {len(history['train_acc'])} epochs in {total_time:.0f}s, best_val={best_val_acc:.2f}%")

    # Cleanup
    del backbone, head, optimizer, scheduler, scaler
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        log_gpu_mem("after cleanup")

    return {
        "model_type": model_type,
        "total_params": total_params,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "final_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "epochs_run": len(history["train_acc"]),
        "total_time_s": total_time,
        "history": history,
    }

log("Training loop defined OK")


## Run All Models

In [ ]:
log_sep("RUNNING ALL MODELS")

all_results = []
total_start = time.time()
TIME_BUDGET_S = 10 * 3600  # 10 hours

for model_idx, (model_type, builder_key, overrides, batch_size) in enumerate(MODELS):
    log_sep(f"MODEL {model_idx+1}/{len(MODELS)}: {model_type.upper()}")

    elapsed = time.time() - total_start
    if elapsed > TIME_BUDGET_S * 0.9:
        log(f"Time budget nearly exhausted ({elapsed/3600:.1f}h / 10h). Skipping.")
        break

    remaining_h = (TIME_BUDGET_S - elapsed) / 3600
    log(f"Elapsed: {elapsed/3600:.1f}h, Remaining budget: {remaining_h:.1f}h")

    try:
        result = train_model(model_type, overrides, batch_size)
        all_results.append(result)
        log(f"{model_type.upper()} completed successfully")
    except Exception as e:
        log(f"ERROR training {model_type}: {e}")
        import traceback; traceback.print_exc()
        all_results.append({"model_type": model_type, "error": str(e)})
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
            log_gpu_mem("after error cleanup")

    # Save intermediate results after each model
    intermediate_path = Path("/kaggle/working/results_intermediate.json")
    intermediate_path.parent.mkdir(parents=True, exist_ok=True)
    serializable_intermediate = []
    for r in all_results:
        d = {k: v for k, v in r.items() if k != "history"}
        serializable_intermediate.append(d)
    with open(intermediate_path, "w") as f:
        json.dump({"results": serializable_intermediate}, f, indent=2)
    log(f"Intermediate results saved to {intermediate_path}")

total_time = time.time() - total_start
log(f"Total training time: {total_time/3600:.2f}h ({total_time:.0f}s)")

## Results Summary

In [ ]:
log_sep("FINAL RESULTS — Tiny-ImageNet Generalization Benchmark")

print("=" * 80, flush=True)
title = f"  {'Model':<10} {'Params':>10} {'Best Val':>10} {'Final Val':>10} {'Gap':>8} {'Epochs':>8} {'Time':>8}"
print(title, flush=True)
print(f"  {'-' * 70}", flush=True)

for r in all_results:
    if "error" in r:
        print(f"  {r['model_type']:<10} ERROR: {r['error']}", flush=True)
        continue
    print(
        f"  {r['model_type']:<10} {r['total_params']/1e6:>9.2f}M "
        f"{r['best_val_acc']:>9.2f}% {r['final_val_acc']:>9.2f}% "
        f"{r['final_gap']:>7.1f}% {r['epochs_run']:>7} {r['total_time_s']:>7.0f}s",
        flush=True,
    )

print(f"\n  Total time: {total_time/3600:.2f}h", flush=True)
print("=" * 80, flush=True)

## Save Results

In [ ]:
log("Saving results to JSON...")
output_path = Path("/kaggle/working/results_tiny_imagenet.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

serializable = []
for r in all_results:
    d = {k: v for k, v in r.items() if k != "history"}
    serializable.append(d)

with open(output_path, "w") as f:
    json.dump({
        "config": {
            "img_size": IMG_SIZE,
            "num_classes": NUM_CLASSES,
            "max_epochs": MAX_EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "seed": SEED,
            "device": str(DEVICE),
            "num_gpus": NUM_GPUS,
        },
        "results": serializable,
    }, f, indent=2)
log(f"Results saved to {output_path}")

## Training Curves

In [ ]:
log("Generating training curves plot...")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for r in all_results:
    if "error" in r or "history" not in r:
        continue
    h = r["history"]
    epochs = list(range(1, len(h["train_acc"]) + 1))

    axes[0].plot(epochs, h["train_acc"], "--", alpha=0.7, label=f"{r['model_type']} train")
    axes[0].plot(epochs, h["val_acc"], "-", alpha=0.9, label=f"{r['model_type']} val")
    axes[0].set_title("Accuracy (%)")
    axes[0].set_xlabel("Epoch")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, h["train_loss"], "--", alpha=0.7, label=f"{r['model_type']} train")
    axes[1].plot(epochs, h["val_loss"], "-", alpha=0.9, label=f"{r['model_type']} val")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    gaps = [h["train_acc"][i] - h["val_acc"][i] for i in range(len(h["train_acc"]))]
    axes[2].plot(epochs, gaps, "-", alpha=0.9, label=r["model_type"])
    axes[2].set_title("Overfitting Gap (train - val)")
    axes[2].set_xlabel("Epoch")
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

plt.suptitle("SpixRWKV-7 Architecture Comparison on Tiny-ImageNet", fontsize=14)
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
log("Saved training_curves.png")

## Architecture Analysis

In [ ]:
log_sep("ARCHITECTURE ANALYSIS")

successful = [r for r in all_results if "error" not in r]
if successful:
    best = max(successful, key=lambda r: r["best_val_acc"])
    most_efficient = min(successful, key=lambda r: r["total_params"])
    fastest = min(successful, key=lambda r: r["total_time_s"])
    least_overfit = min(successful, key=lambda r: r["final_gap"])

    print("=" * 70, flush=True)
    print("  ANALYSIS", flush=True)
    print("=" * 70, flush=True)
    print(f"  Best accuracy:            {best['model_type']} ({best['best_val_acc']:.2f}%)", flush=True)
    print(f"  Most parameter-efficient: {most_efficient['model_type']} ({most_efficient['total_params']/1e6:.2f}M)", flush=True)
    print(f"  Fastest to train:         {fastest['model_type']} ({fastest['total_time_s']:.0f}s)", flush=True)
    print(f"  Least overfitting:        {least_overfit['model_type']} (gap={least_overfit['final_gap']:.1f}%)", flush=True)
    print("=" * 70, flush=True)
else:
    print("No successful runs to analyze.", flush=True)

log("Notebook execution complete!")